In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [ ]:
# Base directories
datasets_dir = Path('../datasets/embeddings')
results_dir = Path('../results')

# Languages to process
languages = ['arabic', 'english']
lang_codes = {'arabic': 'ar', 'english': 'en'}

# Model configurations (model_name: filename_prefix)
models = {
    'BERT': 'bert',
    'RoBERTa': 'roberta',
    'NeoBERT': 'neobert',
    'ModernBERT': 'modernbert',
    'AraBERT-base': 'arabert_base',
    'AraBERT-large': 'arabert_large',
    'BGE-M3': 'bge_m3',
    'E5-large-v2': 'e5_large_v2',
    'Gemma-300M': 'gemma_300m',
    'OpenAI-3-large': 'openai_3_large',
    'OpenAI-ada': 'openai_ada',
    'Voyage-3-large': 'voyage_3_large'
}

print(f"Models to analyze: {len(models)}")
print(f"Languages: {languages}")
print(f"Total configurations: {len(models) * len(languages)}")


In [ ]:
def load_embeddings(filepath):
    """Load embeddings from CSV file and extract embedding columns."""
    df = pd.read_csv(filepath)
    emb_cols = [col for col in df.columns if col.startswith('emb_')]
    return df, emb_cols


def get_embedding_filepath(language, model_prefix):
    """Construct the filepath for a given language and model."""
    lang_code = lang_codes[language]
    filename = f"{model_prefix}_{lang_code}.csv"
    return datasets_dir / language / 'poles' / filename


In [ ]:
def calculate_mean_embeddings(df, emb_cols):
    """Calculate mean embeddings for each VAD dimension and polarity."""
    results = {}
    
    for dimension in ['V', 'A', 'D']:
        for pole in ['pos', 'neg']:
            mask = (df['V/A/D'] == dimension) & (df['pos/neg'] == pole)
            mean_emb = df.loc[mask, emb_cols].mean().values
            results[f'{pole}_{dimension}'] = mean_emb
    
    return results


In [ ]:
def calculate_direction_vectors(mean_embeddings):
    """
    Calculate direction vectors for VAD dimensions.
    Direction = positive_pole - negative_pole
    """
    directions = {}
    
    for dimension in ['V', 'A', 'D']:
        pos_vec = mean_embeddings[f'pos_{dimension}']
        neg_vec = mean_embeddings[f'neg_{dimension}']
        direction = pos_vec - neg_vec
        # Normalize the direction vector
        norm = np.linalg.norm(direction)
        if norm > 0:
            directions[dimension] = direction / norm
        else:
            directions[dimension] = direction
    
    return directions


In [ ]:
def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors."""
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)


In [ ]:
def calculate_orthogonality_metrics(directions):
    """
    Calculate orthogonality metrics between VAD direction vectors.
    Perfect orthogonality = 0 cosine similarity between all pairs.
    """
    v_vec = directions['V']
    a_vec = directions['A']
    d_vec = directions['D']
    
    cos_va = cosine_similarity(v_vec, a_vec)
    cos_vd = cosine_similarity(v_vec, d_vec)
    cos_ad = cosine_similarity(a_vec, d_vec)
    
    abs_cos_va = abs(cos_va)
    abs_cos_vd = abs(cos_vd)
    abs_cos_ad = abs(cos_ad)
    
    mean_abs_cos = (abs_cos_va + abs_cos_vd + abs_cos_ad) / 3
    
    # Orthogonality score: 1 - mean_abs_cos (higher is better, 1.0 = perfect orthogonality)
    orthogonality_score = 1 - mean_abs_cos
    
    return {
        'cos_V_A': cos_va,
        'cos_V_D': cos_vd,
        'cos_A_D': cos_ad,
        'abs_cos_V_A': abs_cos_va,
        'abs_cos_V_D': abs_cos_vd,
        'abs_cos_A_D': abs_cos_ad,
        'mean_abs_cos': mean_abs_cos,
        'orthogonality_score': orthogonality_score
    }


In [ ]:
# Store results per language
results_by_language = {lang: {} for lang in languages}

# Process all models for both languages
for language in languages:
    print(f"\n{'='*60}")
    print(f"Processing {language.upper()} embeddings")
    print(f"{'='*60}")
    
    for model_name, model_prefix in models.items():
        filepath = get_embedding_filepath(language, model_prefix)
        
        if not filepath.exists():
            print(f"  ⚠️  {model_name}: File not found - {filepath}")
            continue
            
        print(f"  Processing {model_name}...")
        
        try:
            df, emb_cols = load_embeddings(filepath)
            mean_embeddings = calculate_mean_embeddings(df, emb_cols)
            directions = calculate_direction_vectors(mean_embeddings)
            orthogonality = calculate_orthogonality_metrics(directions)
            
            results_by_language[language][model_name] = {
                'filepath': str(filepath),
                'embedding_dim': len(emb_cols),
                'num_samples': len(df),
                'mean_embeddings': mean_embeddings,
                'directions': directions,
                'orthogonality': orthogonality
            }
            print(f"    ✓ Dim: {len(emb_cols)}, Samples: {len(df)}, Orthogonality: {orthogonality['orthogonality_score']:.4f}")
        except Exception as e:
            print(f"    ✗ Error: {e}")

print(f"\n{'='*60}")
print("Processing complete!")
print(f"{'='*60}")


Processing BERT...
BERT complete\n
Processing RoBERTa...
RoBERTa complete\n
Processing NeoBERT...
NeoBERT complete\n
Processing ModernBERT...
ModernBERT complete\n


In [ ]:
# Create orthogonality DataFrames per language
orthogonality_dfs = {}

for language in languages:
    print(f"\n{'='*60}")
    print(f"{language.upper()} - Orthogonality Metrics")
    print(f"{'='*60}")
    print("(Lower mean_abs_cos = better orthogonality, Higher orthogonality_score = better)")
    print()
    
    if not results_by_language[language]:
        print("No results available for this language.")
        continue
    
    orthogonality_df = pd.DataFrame({
        model: results['orthogonality']
        for model, results in results_by_language[language].items()
    }).T
    
    # Sort by orthogonality score (higher is better)
    orthogonality_df = orthogonality_df.sort_values('orthogonality_score', ascending=False)
    orthogonality_dfs[language] = orthogonality_df
    
    # Display metrics
    display_cols = ['cos_V_A', 'cos_V_D', 'cos_A_D', 'mean_abs_cos', 'orthogonality_score']
    print(orthogonality_df[display_cols].round(4).to_string())
    print()
    
    # Best model
    best_model = orthogonality_df['orthogonality_score'].idxmax()
    print(f"🏆 Best orthogonality: {best_model}")
    print(f"   Orthogonality score: {orthogonality_df.loc[best_model, 'orthogonality_score']:.4f}")
    print(f"   Mean abs cosine: {orthogonality_df.loc[best_model, 'mean_abs_cos']:.4f}")


Orthogonality Metrics (lower absolute cosine similarity is better):
            cos_V_A  cos_V_D  cos_A_D  abs_cos_V_A  abs_cos_V_D  abs_cos_A_D  \
BERT        -0.1008   0.2354   0.2660       0.1008       0.2354       0.2660   
RoBERTa     -0.0513   0.2413   0.2374       0.0513       0.2413       0.2374   
NeoBERT     -0.2694   0.1213   0.4273       0.2694       0.1213       0.4273   
ModernBERT  -0.1068   0.1981   0.0783       0.1068       0.1981       0.0783   

            mean_abs_cos  
BERT              0.2008  
RoBERTa           0.1767  
NeoBERT           0.2727  
ModernBERT        0.1278  

Best orthogonality: ModernBERT
Mean absolute cosine similarity: 0.1278


In [ ]:
# Save results per language
for language in languages:
    print(f"\n{'='*60}")
    print(f"Saving {language.upper()} results")
    print(f"{'='*60}")
    
    if not results_by_language[language]:
        print("No results to save for this language.")
        continue
    
    lang_results_dir = results_dir / language
    lang_results_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Save direction vectors
    direction_rows = []
    for model_name, results in results_by_language[language].items():
        directions = results['directions']
        for dimension, vector in directions.items():
            row = {
                'model': model_name,
                'dimension': dimension,
                'embedding_dim': results['embedding_dim']
            }
            for i, val in enumerate(vector):
                row[f'dim_{i}'] = val
            direction_rows.append(row)
    
    directions_df = pd.DataFrame(direction_rows)
    directions_path = lang_results_dir / 'direction_vectors.csv'
    directions_df.to_csv(directions_path, index=False)
    print(f"  ✓ Direction vectors saved: {directions_path}")
    print(f"    Rows: {len(directions_df)} (3 dimensions × {len(results_by_language[language])} models)")
    
    # 2. Save orthogonality metrics
    ortho_path = lang_results_dir / 'orthogonality_metrics.csv'
    orthogonality_dfs[language].to_csv(ortho_path)
    print(f"  ✓ Orthogonality metrics saved: {ortho_path}")
    
    # 3. Save mean embeddings (pole embeddings)
    mean_emb_rows = []
    for model_name, results in results_by_language[language].items():
        mean_embeddings = results['mean_embeddings']
        for key, vector in mean_embeddings.items():
            pole, dimension = key.split('_')
            row = {
                'model': model_name,
                'dimension': dimension,
                'pole': pole,
                'embedding_dim': results['embedding_dim']
            }
            for i, val in enumerate(vector):
                row[f'dim_{i}'] = val
            mean_emb_rows.append(row)
    
    mean_emb_df = pd.DataFrame(mean_emb_rows)
    mean_emb_path = lang_results_dir / 'mean_pole_embeddings.csv'
    mean_emb_df.to_csv(mean_emb_path, index=False)
    print(f"  ✓ Mean pole embeddings saved: {mean_emb_path}")
    print(f"    Rows: {len(mean_emb_df)} (6 poles × {len(results_by_language[language])} models)")

print(f"\n{'='*60}")
print("All results saved successfully!")
print(f"{'='*60}")


Results saved to ../Datasets/orthogonality_analysis_results.csv
Total rows: 40


In [ ]:
# Summary and comparison between languages
print("="*70)
print("FINAL SUMMARY: Orthogonality Analysis Results")
print("="*70)

print(f"\nModels analyzed: {len(models)}")
print(f"Languages: {', '.join(languages)}")
print(f"Total configurations: {sum(len(r) for r in results_by_language.values())}")

# Per-language rankings
for language in languages:
    if language not in orthogonality_dfs or orthogonality_dfs[language].empty:
        continue
        
    print(f"\n{'─'*70}")
    print(f"📊 {language.upper()} Rankings (by orthogonality score, higher = better)")
    print(f"{'─'*70}")
    
    ranking = orthogonality_dfs[language].sort_values('orthogonality_score', ascending=False)
    for i, (model, row) in enumerate(ranking.iterrows(), 1):
        emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
        print(f"{emoji} {i:2d}. {model:<20} | Score: {row['orthogonality_score']:.4f} | Mean|cos|: {row['mean_abs_cos']:.4f}")

# Cross-language comparison
print(f"\n{'─'*70}")
print("🌐 Cross-Language Best Models Comparison")
print(f"{'─'*70}")

for language in languages:
    if language not in orthogonality_dfs or orthogonality_dfs[language].empty:
        continue
    best = orthogonality_dfs[language]['orthogonality_score'].idxmax()
    score = orthogonality_dfs[language].loc[best, 'orthogonality_score']
    print(f"  {language.capitalize()}: {best} (score: {score:.4f})")

print(f"\n{'='*70}")
print("Analysis complete. Results saved to ../results/{language}/ directories.")
print("="*70)


Summary:
Total models analyzed: 4
Best model for orthogonality: ModernBERT
\nOrthogonality ranking (lower is better):
1. ModernBERT: 0.1278
2. RoBERTa: 0.1767
3. BERT: 0.2008
4. NeoBERT: 0.2727


In [ ]:
# Display saved files structure
print("Output files structure:")
for language in languages:
    lang_results_dir = results_dir / language
    if lang_results_dir.exists():
        print(f"\n{language}/")
        for f in sorted(lang_results_dir.glob("*.csv")):
            size_kb = f.stat().st_size / 1024
            print(f"  └── {f.name} ({size_kb:.1f} KB)")
